In [12]:
# Client Setup
import boto3

client = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

In [3]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [4]:
# Tool Schemas

article_details_schema = {
    "toolSpec": {
        "name": "article_details",
        "description": "This tool should be called with details about an article. It accepts information about the article's title, author, and related topics.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article. Can be left empty.",
                    },
                    "author": {
                        "type": "string",
                        "description": "The name of the article's author. Can be left empty.",
                    },
                    "topics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A list of topics or categories that the article covers. Can be an empty list.",
                    },
                },
            }
        },
    }
}

to_json_schema = {
    "toolSpec": {
        "name": "to_json",
        "description": "This tool processes any JSON data and can be used for generating structured content, transforming information, or creating any JSON-based output needed for your task.",
        "inputSchema": {
            "json": {"type": "object", "additionalProperties": True}
        },
    }
}

In [5]:
messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["text"])

result["text"]

'# The Evolution of Machine Learning Paradigms: From Symbolic AI to Deep Neural Networks\n\n**Dr. Margaret Chen, Department of Computer Science, Stanford University**\n\nThe field of computer science has undergone a fundamental transformation in its approach to artificial intelligence, transitioning from early symbolic and rule-based systems to contemporary deep learning methodologies that leverage neural networks and massive computational resources. This paradigm shift, which gained significant momentum following the widespread availability of big data and GPU acceleration in the early 2010s, has redefined our understanding of what constitutes effective machine learning, replacing the labor-intensive process of manual feature engineering with hierarchical feature extraction learned directly from raw data. While symbolic AI systems of the 1970s and 1980s operated on explicit human-defined rules and struggled with tasks requiring contextual understanding, modern deep learning architectu

In [6]:
messages = []

add_user_message(messages, f"""Analyze the article below and extract key data. Then call the article_details tool.

<article_text>
{result["text"]}
</article_text>

 """)


json_result = chat(messages, tools=[article_details_schema], tool_choice= "article_details")


json_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_iHtIqQVVXu547AqlYTF4ij',
    'name': 'article_details',
    'input': {'title': 'The Evolution of Machine Learning Paradigms: From Symbolic AI to Deep Neural Networks',
     'author': 'Dr. Margaret Chen',
     'topics': ['Machine Learning',
      'Artificial Intelligence',
      'Symbolic AI',
      'Deep Learning',
      'Neural Networks',
      'Computer Vision',
      'Natural Language Processing',
      'Feature Engineering',
      'Hybrid AI',
      'GPU Acceleration']},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}

In [10]:
messages = []

add_user_message(messages,f"""Analyze the article below and extract key data. Then call the to_json tool.

<article_text>
{result["text"]}
</article_text>

When you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics in the article,
    "number_topics": int # Number of topics mentioned}}""")

flexbible_result = chat(messages, tools = [to_json_schema], tool_choice="to_json")

In [11]:
flexbible_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_5NYkPGnBIYj8BaqnjCGD3B',
    'name': 'to_json',
    'input': {'title': 'The Evolution of Machine Learning Paradigms: From Symbolic AI to Deep Neural Networks',
     'author': 'Dr. Margaret Chen, Department of Computer Science, Stanford University',
     'topics': ['Symbolic AI',
      'Rule-based Systems',
      'Deep Learning',
      'Neural Networks',
      'Feature Engineering',
      'Big Data',
      'GPU Acceleration',
      'Convolutional Neural Networks',
      'Recurrent Neural Networks',
      'Natural Language Processing',
      'Computer Vision',
      'Interpretability',
      'Hybrid AI Approaches'],
     'number_topics': 13},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}